In [1]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.ops import unary_union

from pathlib import Path
from tqdm.auto import tqdm

datasets = Path("/nas/cee-ice/data")

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subbasins = gpd.read_parquet(metadata_dir / 'subbasins.parquet')
subbasins.index = subbasins.index.astype(str)
subbasins.geometry = subbasins.geometry.centroid
footprint_geom = unary_union(subbasins.geometry.envelope)

/tmp/ipykernel_428341/4284604122.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subbasins.geometry = subbasins.geometry.centroid


In [2]:
subbasins

,geometry,area_km2,uparea_km2,node_type,is_gauge,nextdown,outlet_id
site_id,,,,,,,
23021321,POINT (9.158 46.94895),209.123678,209.123678,merged,False,23017718,EAUF-A3750050
23021261,POINT (9.26107 46.53043),213.109547,213.109547,merged,False,23020936,EAUF-A3750050
23021244,POINT (9.82102 46.74299),291.030672,291.030672,merged,False,23021024,EAUF-A3750050
23021122,POINT (8.54697 46.663),477.964586,477.964586,merged,False,23020916,EAUF-A3750050
23021112,POINT (8.81039 46.66715),394.295017,394.295017,merged,False,23020913,EAUF-A3750050
...,...,...,...,...,...,...,...
81003765,POINT (-149.03954 68.37231),497.326177,1613.152478,merged,False,81001413,USGS-15908000
81001414,POINT (-148.53207 68.6856),166.763202,1473.949940,merged,False,81001326,USGS-15908000
81001413,POINT (-148.91291 68.62543),371.141787,2751.705206,merged,False,81001326,USGS-15908000


In [3]:
# Read in hydroatlas 
gdb_path = datasets / "HydroSHEDS" / "HydroATLAS" / "BasinATLAS_v10.gdb"
layer_name = "BasinATLAS_v10_lev09"

hydroatlas = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio", use_arrow=True)
hydroatlas.replace({'wet_cl_smj':-9999}, 13, inplace=True)
hydroatlas = hydroatlas.set_index('HYBAS_ID')

# hydroatlas_filt = hydroatlas[hydroatlas.intersects(footprint_geom)]
# hydroatlas_filt

/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/pyogrio/raw.py:337: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  table = reader.read_all()


In [ ]:
hydroatlas_filt = hydroatlas[hydroatlas.intersects(footprint_geom)]
hydroatlas_filt

In [4]:
hydroatlas_filt = hydroatlas

In [5]:
subs_buff = subbasins.copy()
subs_buff = subs_buff.to_crs("EPSG:6933")
subs_buff.geometry = subs_buff.buffer(2500)

hybas_pairs = gpd.sjoin(
    subs_buff,
    hydroatlas_filt.to_crs("EPSG:6933"),
    how='left',
)

hybas_area = hybas_pairs['UP_AREA']
merit_area = hybas_pairs['uparea_km2']
hybas_pairs['hybas_area_diff'] = (hybas_area - merit_area) / ((hybas_area + merit_area)/2)

def get_best_hybas(grp):
     # Pick best match by area.
    idx_min = grp['hybas_area_diff'].abs().idxmin()
    return grp.loc[idx_min]

all_attributes = hybas_pairs.reset_index().groupby('site_id').apply(get_best_hybas, include_groups=False)

In [6]:
attributes_dir = save_dir / 'attributes'

In [7]:
hybas_properties = pd.read_csv(attributes_dir / "properties.txt", header=None)[0].to_list()
matchup_props = ['outlet_id', 'area_km2','uparea_km2', 'HYBAS_ID']
all_props = matchup_props + hybas_properties + ['hybas_area_diff']

In [8]:
attributes = (
    all_attributes[all_props]
    .reset_index()
    .rename(columns={'site_id':'subbasin', 'outlet_id': 'basin'})
    .astype({'subbasin':str, 'basin': str})
    .set_index(['basin', 'subbasin'])
)
attributes

area_km2   uparea_km2      HYBAS_ID  \
basin         subbasin                                               
EAUF-V7200010 21000002       468.102025  4364.328752  2.090493e+09   
              21000020       401.905725  3625.201976  2.090490e+09   
              21000022       292.309508  3076.492166  2.090488e+09   
              21000026       481.039377  1767.807919  2.090486e+09   
              21000032       390.920772   621.770285  2.090484e+09   
...                                 ...          ...           ...   
USGS-50129254 USGS-50129254   42.674493    42.674493  7.090073e+09   
USGS-50138000 USGS-50136400   51.402873    51.402873  7.090073e+09   
              USGS-50138000  261.453879   312.856752  7.090073e+09   
USGS-50144000 USGS-50144000  349.557308   349.557308  7.090073e+09   
USGS-50147800 USGS-50147800  195.872993   195.872993  7.090073e+09   

                             dis_m3_pyr  dis_m3_pmn  dis_m3_pmx  run_mm_syr  \
basin         subbasin                                                        
EAUF-V7200010 21000002        66.342003   22.684999  108.996002         525   
              21000020        48.854000   16.743000   80.602997         537   
              21000022        28.339001    9.780000   46.625000         504   
              21000026         6.031000    2.077000    9.826000         422   
              21000032         8.653000    3.024000   14.126000         437   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254    4.198000    1.576000    8.597000         578   
USGS-50138000 USGS-50136400    0.185000    0.068000    0.359000         598   
              USGS-50138000    5.788000    2.068000   11.577000         593   
USGS-50144000 USGS-50144000    9.208000    3.183000   19.839001         609   
USGS-50147800 USGS-50147800    5.115000    1.855000   10.214000         601   

                             inu_pc_smn  inu_pc_smx  inu_pc_slt  ...  \
basin         subbasin                                           ...   
EAUF-V7200010 21000002                0           0           1  ...   
              21000020                0           0           0  ...   
              21000022                0           0           0  ...   
              21000026                0           0           0  ...   
              21000032                0           0           0  ...   
...                                 ...         ...         ...  ...   
USGS-50129254 USGS-50129254          61          69          72  ...   
USGS-50138000 USGS-50136400          63          79          80  ...   
              USGS-50138000          30          48          51  ...   
USGS-50144000 USGS-50144000          36          61          62  ...   
USGS-50147800 USGS-50147800          26          38          38  ...   

                             urb_pc_sse  nli_ix_sav  rdd_mk_sav  hft_ix_s93  \
basin         subbasin                                                        
EAUF-V7200010 21000002                0         844         711         135   
              21000020                0        1754         786         187   
              21000022                0         842         631         133   
              21000026                0         728         759         142   
              21000032                0         824         536         127   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254          22        3209        2365         296   
USGS-50138000 USGS-50136400          48        4778        5447         363   
              USGS-50138000          27        3680        2432         291   
USGS-50144000 USGS-50144000           4        1686        1292         222   
USGS-50147800 USGS-50147800          39        3856        1703         301   

                             hft_ix_s09  gad_id_smj  gdp_ud_sav    gdp_ud_ssu  \
basin         subbasin                        

In [37]:
gdw = gpd.read_file('/nas/cee-water/cjgleason/ted/GDW/GDW_v1_0_shp/GDW_reservoirs_v1_0.shp')

In [65]:
df3.geometry.area

0       6.737596e+04
1       3.072371e+05
2       7.499817e+05
3       1.820753e+06
4       4.791478e+06
            ...     
2490    1.497915e+06
2491    1.064073e+06
2492    4.509541e+05
2493    2.460119e+06
2494    1.272827e+06
Length: 2495, dtype: float64

In [67]:
# df3['area_overlap'] = df3.geometry.area
# df3.sort_values(by='area_overlap', inplace=True)
df3.drop_duplicates(subset='GDW_ID', keep='last', inplace=True)
df3

,area_km2,uparea_km2,node_type,is_gauge,nextdown,outlet_id,GDW_ID,RES_NAME,DAM_NAME,ALT_NAME,...,LAT_DAM,ORIG_SRC,POLY_SRC,GRAND_ID,HYRIV_ID,INSTREAM,HYLAK_ID,HYBAS_L12,geometry,area_overlap
1711,404.456784,876.330305,merged,False,74038728,USGS-07374000,11322,None,MACOUPIN ENERGY SMITH RESERVOIR DAM,SMITH RESERVOIR,...,0.0,GOODD-NID,JRC-GSW,0,70589896,Offstream,1056946,7121080830,"POLYGON ((-8672045.815 4629704.743, -8672047.4...",4.790285e+00
269,488.745565,1826.304232,merged,False,62140619,BRANA-29700000,13335,None,None,None,...,0.0,GOODD,SWBD,0,61194851,Offstream,1094439,6121205550,"POLYGON ((-4716715.311 -1915828.586, -4716706....",2.072476e+01
2249,260.972716,471947.730072,gauge,True,USGS-09429600,USGS-09521100,27468,None,IMPERIAL DIVERSION,None,...,0.0,GOODD-NID,JRC-GSW-mod,0,70746486,Instream,1430761,7121144020,"POLYGON ((-11044110.437 3981207.653, -11044107...",2.531454e+01
334,74.762374,6566.375572,merged,False,63007593,BRANA-34879500,12851,None,None,None,...,0.0,GOODD,JRC-GSW,0,60741322,Offstream,1086595,6120336520,"POLYGON ((-3931384.583 -658722.514, -3931375.7...",2.602557e+01
2067,397.982013,397.982013,merged,False,74062450,USGS-07381490,11689,None,SCS-UPPER ELK CREEK SITE-22R,ELK CITY LAKE,...,0.0,GOODD-NID,SWBD,0,70685279,Instream,1060455,7121119600,"POLYGON ((-9591401.801 4236014.771, -9591406.3...",1.329516e+02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1053,481.859490,7857.932569,merged,False,72029770,ECCC-02OA016,16,Nipigon,Pine Portage Main,None,...,0.0,GRanD,CanVec,1402,70283953,Instream,58,7120302220,"POLYGON ((-8516426.386 5597071.482, -8516462.3...",1.960343e+07
1247,433.500788,4939.572071,merged,False,71002544,ECCC-05TG001,5,Southern Indian,Missi Falls Control,Notigi Control,...,0.0,GRanD,Other,702,70037207,Instream,37,7120896070,"POLYGON ((-9491873.313 6173800.899, -9491909.3...",1.960343e+07
1531,386.088953,386.088953,gauge,True,None,USGS-02172002,172,Lake Moultrie,Pinopolis West Dam,None,...,0.0,GRanD,SWBD,1890,70738014,Instream,830,7120698290,"POLYGON ((-7722232.516 4019650.256, -7722268.5...",1.960343e+07
110,481.644724,45314.159774,merged,False,56007618,ABOM-128547010,398,Argyle,Ord River,None,...,0.0,GRanD,SWBD,6201,50472071,Instream,1632,5120246810,"POLYGON ((12428761.802 -2051701.837, 12428725....",1.960343e+07


In [71]:
gdw_pairs

,area_km2,uparea_km2,node_type,is_gauge,nextdown,outlet_id,GDW_ID,RES_NAME,DAM_NAME,ALT_NAME,...,LAT_DAM,ORIG_SRC,POLY_SRC,GRAND_ID,HYRIV_ID,INSTREAM,HYLAK_ID,HYBAS_L12,geometry,area_overlap
1711,404.456784,876.330305,merged,False,74038728,USGS-07374000,11322,None,MACOUPIN ENERGY SMITH RESERVOIR DAM,SMITH RESERVOIR,...,0.0,GOODD-NID,JRC-GSW,0,70589896,Offstream,1056946,7121080830,"POLYGON ((-8672045.815 4629704.743, -8672047.4...",4.790285e+00
269,488.745565,1826.304232,merged,False,62140619,BRANA-29700000,13335,None,None,None,...,0.0,GOODD,SWBD,0,61194851,Offstream,1094439,6121205550,"POLYGON ((-4716715.311 -1915828.586, -4716706....",2.072476e+01
2249,260.972716,471947.730072,gauge,True,USGS-09429600,USGS-09521100,27468,None,IMPERIAL DIVERSION,None,...,0.0,GOODD-NID,JRC-GSW-mod,0,70746486,Instream,1430761,7121144020,"POLYGON ((-11044110.437 3981207.653, -11044107...",2.531454e+01
334,74.762374,6566.375572,merged,False,63007593,BRANA-34879500,12851,None,None,None,...,0.0,GOODD,JRC-GSW,0,60741322,Offstream,1086595,6120336520,"POLYGON ((-3931384.583 -658722.514, -3931375.7...",2.602557e+01
2067,397.982013,397.982013,merged,False,74062450,USGS-07381490,11689,None,SCS-UPPER ELK CREEK SITE-22R,ELK CITY LAKE,...,0.0,GOODD-NID,SWBD,0,70685279,Instream,1060455,7121119600,"POLYGON ((-9591401.801 4236014.771, -9591406.3...",1.329516e+02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1053,481.859490,7857.932569,merged,False,72029770,ECCC-02OA016,16,Nipigon,Pine Portage Main,None,...,0.0,GRanD,CanVec,1402,70283953,Instream,58,7120302220,"POLYGON ((-8516426.386 5597071.482, -8516462.3...",1.960343e+07
1247,433.500788,4939.572071,merged,False,71002544,ECCC-05TG001,5,Southern Indian,Missi Falls Control,Notigi Control,...,0.0,GRanD,Other,702,70037207,Instream,37,7120896070,"POLYGON ((-9491873.313 6173800.899, -9491909.3...",1.960343e+07
1531,386.088953,386.088953,gauge,True,None,USGS-02172002,172,Lake Moultrie,Pinopolis West Dam,None,...,0.0,GRanD,SWBD,1890,70738014,Instream,830,7120698290,"POLYGON ((-7722232.516 4019650.256, -7722268.5...",1.960343e+07
110,481.644724,45314.159774,merged,False,56007618,ABOM-128547010,398,Argyle,Ord River,None,...,0.0,GRanD,SWBD,6201,50472071,Instream,1632,5120246810,"POLYGON ((12428761.802 -2051701.837, 12428725....",1.960343e+07


In [79]:
gdw_pairs

,,index,area_km2,uparea_km2,node_type,is_gauge,nextdown,GDW_ID,RES_NAME,DAM_NAME,ALT_NAME,...,LAT_DAM,ORIG_SRC,POLY_SRC,GRAND_ID,HYRIV_ID,INSTREAM,HYLAK_ID,HYBAS_L12,geometry,area_overlap
basin,subbasin,,,,,,,,,,,,,,,,,,,,,
USGS-07374000,74039029,1711,404.456784,876.330305,merged,False,74038728,11322,None,MACOUPIN ENERGY SMITH RESERVOIR DAM,SMITH RESERVOIR,...,0.0,GOODD-NID,JRC-GSW,0,70589896,Offstream,1056946,7121080830,"POLYGON ((-8672045.815 4629704.743, -8672047.4...",4.790285e+00
BRANA-29700000,62140821,269,488.745565,1826.304232,merged,False,62140619,13335,None,None,None,...,0.0,GOODD,SWBD,0,61194851,Offstream,1094439,6121205550,"POLYGON ((-4716715.311 -1915828.586, -4716706....",2.072476e+01
USGS-09521100,USGS-09429500,2249,260.972716,471947.730072,gauge,True,USGS-09429600,27468,None,IMPERIAL DIVERSION,None,...,0.0,GOODD-NID,JRC-GSW-mod,0,70746486,Instream,1430761,7121144020,"POLYGON ((-11044110.437 3981207.653, -11044107...",2.531454e+01
BRANA-34879500,63007599,334,74.762374,6566.375572,merged,False,63007593,12851,None,None,None,...,0.0,GOODD,JRC-GSW,0,60741322,Offstream,1086595,6120336520,"POLYGON ((-3931384.583 -658722.514, -3931375.7...",2.602557e+01
USGS-07381490,74062928,2067,397.982013,397.982013,merged,False,74062450,11689,None,SCS-UPPER ELK CREEK SITE-22R,ELK CITY LAKE,...,0.0,GOODD-NID,SWBD,0,70685279,Instream,1060455,7121119600,"POLYGON ((-9591401.801 4236014.771, -9591406.3...",1.329516e+02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ECCC-02OA016,72029773,1053,481.859490,7857.932569,merged,False,72029770,16,Nipigon,Pine Portage Main,None,...,0.0,GRanD,CanVec,1402,70283953,Instream,58,7120302220,"POLYGON ((-8516426.386 5597071.482, -8516462.3...",1.960343e+07
ECCC-05TG001,71002647,1247,433.500788,4939.572071,merged,False,71002544,5,Southern Indian,Missi Falls Control,Notigi Control,...,0.0,GRanD,Other,702,70037207,Instream,37,7120896070,"POLYGON ((-9491873.313 6173800.899, -9491909.3...",1.960343e+07
USGS-02172002,USGS-02172002,1531,386.088953,386.088953,gauge,True,None,172,Lake Moultrie,Pinopolis West Dam,None,...,0.0,GRanD,SWBD,1890,70738014,Instream,830,7120698290,"POLYGON ((-7722232.516 4019650.256, -7722268.5...",1.960343e+07


In [81]:
gdw_pairs = gpd.overlay(subs_buff.reset_index(), gdw.to_crs("EPSG:6933"), how='intersection')

#Sort by area so largest area is last
gdw_pairs['area_overlap'] = gdw_pairs.geometry.area

gdw_pairs = (
    gdw_pairs
    .sort_values(by='area_overlap')
    .reset_index()
    .rename(columns={'site_id':'subbasin', 'outlet_id': 'basin'})
    .astype({'subbasin':str, 'basin': str})
    .drop_duplicates(subset='subbasin', keep='last')
    .set_index(['basin', 'subbasin'])
)
gdw_pairs

,,index,area_km2,uparea_km2,node_type,is_gauge,nextdown,GDW_ID,RES_NAME,DAM_NAME,ALT_NAME,...,LAT_DAM,ORIG_SRC,POLY_SRC,GRAND_ID,HYRIV_ID,INSTREAM,HYLAK_ID,HYBAS_L12,geometry,area_overlap
basin,subbasin,,,,,,,,,,,,,,,,,,,,,
USGS-07374000,74039029,1711,404.456784,876.330305,merged,False,74038728,11322,None,MACOUPIN ENERGY SMITH RESERVOIR DAM,SMITH RESERVOIR,...,0.0,GOODD-NID,JRC-GSW,0,70589896,Offstream,1056946,7121080830,"POLYGON ((-8672045.815 4629704.743, -8672047.4...",4.790285e+00
BRANA-85900000,64054856,890,403.790635,13143.483742,merged,False,64061731,1202,None,Dona Francisca,None,...,0.0,GRanD,McGill,2453,61449523,Instream,10431,6120842380,"POLYGON ((-5135185.686 -3587760.276, -5135184....",1.160045e+01
BRANA-29700000,62140821,269,488.745565,1826.304232,merged,False,62140619,13335,None,None,None,...,0.0,GOODD,SWBD,0,61194851,Offstream,1094439,6121205550,"POLYGON ((-4716715.311 -1915828.586, -4716706....",2.072476e+01
USGS-09521100,USGS-09429500,2249,260.972716,471947.730072,gauge,True,USGS-09429600,27468,None,IMPERIAL DIVERSION,None,...,0.0,GOODD-NID,JRC-GSW-mod,0,70746486,Instream,1430761,7121144020,"POLYGON ((-11044110.437 3981207.653, -11044107...",2.531454e+01
BRANA-34879500,63007599,334,74.762374,6566.375572,merged,False,63007593,12851,None,None,None,...,0.0,GOODD,JRC-GSW,0,60741322,Offstream,1086595,6120336520,"POLYGON ((-3931384.583 -658722.514, -3931375.7...",2.602557e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-02172002,USGS-02172002,1531,386.088953,386.088953,gauge,True,None,172,Lake Moultrie,Pinopolis West Dam,None,...,0.0,GRanD,SWBD,1890,70738014,Instream,830,7120698290,"POLYGON ((-7722232.516 4019650.256, -7722268.5...",1.960343e+07
ABOM-128547010,56007621,110,481.644724,45314.159774,merged,False,56007618,398,Argyle,Ord River,None,...,0.0,GRanD,SWBD,6201,50472071,Instream,1632,5120246810,"POLYGON ((12428761.802 -2051701.837, 12428725....",1.960343e+07
BRANA-49705000,63018724,534,460.798854,492613.797720,merged,False,63018720,27,Itaparica,Sobradinho Main Dam,Sobradinho Reservoir; Bsobrandinho,...,0.0,GRanD,McGill,2516,60975854,Instream,77,6120464720,"POLYGON ((-4000925.321 -1227031.959, -4000961....",1.960343e+07


In [82]:
gdw_props = ["DAM_TYPE", "YEAR_DAM", "AREA_POLY", "CAP_MCM", "DIS_AVG_LS", "DOR_PC"]
expanded_attr = attributes.join(gdw_pairs[gdw_props])
expanded_attr['has_dam'] = ~expanded_attr['DAM_TYPE'].isna()

expanded_attr

area_km2   uparea_km2      HYBAS_ID  \
basin         subbasin                                               
EAUF-V7200010 21000002       468.102025  4364.328752  2.090493e+09   
              21000020       401.905725  3625.201976  2.090490e+09   
              21000022       292.309508  3076.492166  2.090488e+09   
              21000026       481.039377  1767.807919  2.090486e+09   
              21000032       390.920772   621.770285  2.090484e+09   
...                                 ...          ...           ...   
USGS-50129254 USGS-50129254   42.674493    42.674493  7.090073e+09   
USGS-50138000 USGS-50136400   51.402873    51.402873  7.090073e+09   
              USGS-50138000  261.453879   312.856752  7.090073e+09   
USGS-50144000 USGS-50144000  349.557308   349.557308  7.090073e+09   
USGS-50147800 USGS-50147800  195.872993   195.872993  7.090073e+09   

                             dis_m3_pyr  dis_m3_pmn  dis_m3_pmx  run_mm_syr  \
basin         subbasin                                                        
EAUF-V7200010 21000002        66.342003   22.684999  108.996002         525   
              21000020        48.854000   16.743000   80.602997         537   
              21000022        28.339001    9.780000   46.625000         504   
              21000026         6.031000    2.077000    9.826000         422   
              21000032         8.653000    3.024000   14.126000         437   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254    4.198000    1.576000    8.597000         578   
USGS-50138000 USGS-50136400    0.185000    0.068000    0.359000         598   
              USGS-50138000    5.788000    2.068000   11.577000         593   
USGS-50144000 USGS-50144000    9.208000    3.183000   19.839001         609   
USGS-50147800 USGS-50147800    5.115000    1.855000   10.214000         601   

                             inu_pc_smn  inu_pc_smx  inu_pc_slt  ...  \
basin         subbasin                                           ...   
EAUF-V7200010 21000002                0           0           1  ...   
              21000020                0           0           0  ...   
              21000022                0           0           0  ...   
              21000026                0           0           0  ...   
              21000032                0           0           0  ...   
...                                 ...         ...         ...  ...   
USGS-50129254 USGS-50129254          61          69          72  ...   
USGS-50138000 USGS-50136400          63          79          80  ...   
              USGS-50138000          30          48          51  ...   
USGS-50144000 USGS-50144000          36          61          62  ...   
USGS-50147800 USGS-50147800          26          38          38  ...   

                               gdp_ud_ssu  hdi_ix_sav  hybas_area_diff  \
basin         subbasin                                                   
EAUF-V7200010 21000002       3.976704e+08         910        -0.059877   
              21000020       1.899765e+08         910        -0.164848   
              21000022       1.372329e+08         910        -0.440938   
              21000026       2.234216e+08         899        -1.183507   
              21000032       2.993196e+08         902         0.015052   
...                                   ...         ...              ...   
USGS-50129254 USGS-50129254  1.574034e+09         832         1.339595   
USGS-50138000 USGS-50136400  1.705076e+09         832         0.288039   
              USGS-50138000  3.784680e+09         832         0.085791   
USGS-50144000 USGS-50144000  1.818283e+09         832         0.295377   
USGS-50147800 USGS-50147800  3.385777e+09         832         0.399315   

                             DAM_TYPE  YEAR_DAM  AREA_POLY  CAP_MCM  \
basin         subbasin                                                
EAUF-V7200010 21000002            NaN       NaN   

In [85]:
expanded_attr.to_parquet(attributes_dir / f"static.parquet")

In [36]:
test = pd.read_parquet(attributes_dir / "static.parquet")

In [37]:
test

area_km2   uparea_km2  dis_m3_pyr  dis_m3_pmn  \
basin         subbasin                                                         
EAUF-V7200010 21000002       468.102025  4364.328752   66.342003   22.684999   
              21000020       401.905725  3625.201976   48.854000   16.743000   
              21000022       292.309508  3076.492166   28.339001    9.780000   
              21000026       481.039377  1767.807919    6.031000    2.077000   
              21000032       390.920772   621.770285    8.653000    3.024000   
...                                 ...          ...         ...         ...   
USGS-50129254 USGS-50129254   42.674493    42.674493    4.198000    1.576000   
USGS-50138000 USGS-50136400   51.402873    51.402873    0.185000    0.068000   
              USGS-50138000  261.453879   312.856752    5.788000    2.068000   
USGS-50144000 USGS-50144000  349.557308   349.557308    9.208000    3.183000   
USGS-50147800 USGS-50147800  195.872993   195.872993    5.115000    1.855000   

                             dis_m3_pmx  run_mm_syr  inu_pc_smn  inu_pc_smx  \
basin         subbasin                                                        
EAUF-V7200010 21000002       108.996002         525           0           0   
              21000020        80.602997         537           0           0   
              21000022        46.625000         504           0           0   
              21000026         9.826000         422           0           0   
              21000032        14.126000         437           0           0   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254    8.597000         578          61          69   
USGS-50138000 USGS-50136400    0.359000         598          63          79   
              USGS-50138000   11.577000         593          30          48   
USGS-50144000 USGS-50144000   19.839001         609          36          61   
USGS-50147800 USGS-50147800   10.214000         601          26          38   

                             inu_pc_slt  lka_pc_sse  ...  urb_pc_sse  \
basin         subbasin                               ...               
EAUF-V7200010 21000002                1           0  ...           0   
              21000020                0           0  ...           0   
              21000022                0           1  ...           0   
              21000026                0           0  ...           0   
              21000032                0           0  ...           0   
...                                 ...         ...  ...         ...   
USGS-50129254 USGS-50129254          72           1  ...          22   
USGS-50138000 USGS-50136400          80           0  ...          48   
              USGS-50138000          51           0  ...          27   
USGS-50144000 USGS-50144000          62           1  ...           4   
USGS-50147800 USGS-50147800          38           0  ...          39   

                             nli_ix_sav  rdd_mk_sav  hft_ix_s93  hft_ix_s09  \
basin         subbasin                                                        
EAUF-V7200010 21000002              844         711         135         106   
              21000020             1754         786         187         163   
              21000022              842         631         133          88   
              21000026              728         759         142          92   
              21000032              824         536         127          86   
...                                 ...         ...         ...         ...   
USGS-50129254 USGS-50129254        3209        2365         296         277   
USGS-50138000 USGS-50136400        4778        5447         363         346   
              USGS-50138000        3680        2432         291         288   
USGS-50144000 USGS-50144000        1686        1292         222         173   
USGS-50147800 USGS-50147800        3856        1703         301         286  